# Учительская проверка submission: логистическая регрессия

Этот блокнот не нужно раздавать участникам вместе с ответами. Он читает файлы команд из папки `submissions` и сравнивает их со скрытым `data/private_target.csv`.


In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

DATA_DIR = Path("data")
SUBMISSIONS_DIR = Path("submissions")

private = pd.read_csv(DATA_DIR / "private_target.csv")
files = sorted(SUBMISSIONS_DIR.glob("*.csv"))

rows = []
for path in files:
    sub = pd.read_csv(path)
    merged = private.merge(sub, on="id", how="inner", suffixes=("_true", "_pred"))
    if len(merged) != len(private):
        print(f"Пропускаю {path.name}: найдено {len(merged)} id из {len(private)}")
        continue
    if "will_finish_pred" in merged.columns:
        y_pred = merged["will_finish_pred"].astype(int)
    elif "probability" in merged.columns:
        y_pred = (merged["probability"] >= 0.5).astype(int)
    else:
        print(f"Пропускаю {path.name}: нужен столбец will_finish или probability")
        continue
    y_true_col = "will_finish_true" if "will_finish_true" in merged.columns else "will_finish"
    y_true = merged[y_true_col]
    row = {
        "team_file": path.name,
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
    }
    if "probability" in merged.columns:
        row["ROC_AUC"] = roc_auc_score(y_true, merged["probability"])
    rows.append(row)

if rows:
    leaderboard = pd.DataFrame(rows).sort_values(["F1", "recall"], ascending=[False, False])
    display(leaderboard)
else:
    print("Положите CSV-файлы команд в папку submissions и запустите ячейку снова.")
